## 1.  Import Libraries



# Exploring Rhymes in the Du Chemin Chansons Nouvelles Texts via TEI

- In this notebook we will work with a TEI encoding of all the texts in the Du Chemin Chansons Nouvelles corpus, which forms the basis of the Lost Voice Project (https://digitalduchemin.org). We will parse the TEI XML to extract the lines of each piece, along with their rhyme schemes, and then build a network graph of the rhymes across the corpus.

In [1]:
import pandas as pd
from pyvis.network import Network
from itertools import combinations
import networkx as nx
import re
from lxml import etree
from community import community_louvain
from copy import deepcopy

# plot libraries and settings for Jupyter notebooks and Quarto
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook"

## 2.  Import the Data from TEI



In [2]:
# names spaces and functions

NS = 'http://www.tei-c.org/ns/1.0'
XML_NS = 'http://www.w3.org/XML/1998/namespace'

def get_line_text(l_elem):
    """Extract line text, substituting <expan> for <abbr> inside <choice>."""
    parts = []
    if l_elem.text:
        parts.append(l_elem.text)
    for child in l_elem:
        local = child.tag.split('}')[-1] if '}' in child.tag else child.tag
        if local == 'choice':
            expan = child.find(f'{{{NS}}}expan')
            if expan is not None and expan.text:
                parts.append(expan.text)
        else:
            if child.text:
                parts.append(child.text)
        if child.tail:
            parts.append(child.tail)
    return ''.join(parts).strip()



- What does a single poem look like in the TEI XML? -

- What are the relevant tags for parsing the lines, rhyme schemes, and meters?

```python
<div xml:id="DC0101" type="poem" rhyme="ababbcbc" met="10">

                <lg>
                    <l enjamb="weak">Qui souhaitez avoir tout le plaisir</l>
                    <l>Qu'un amy peult vouloir honnestement</l>
                    <l>Prenez exemple à mon chaste desir</l>
                    <l>Et vous mirez en mon contentement.</l>
                </lg>
                <lg>
                    <l enjamb="strong">Mais qui vouldroit audacieusement</l>
                    <l>Voler au ciel ou mon amour se tient,</l>
                    <l>On luy diroit aymez humainement.</l>
                    <l>C'est au soleil que la lune appartient.</l>
                </lg>
            </div>
```

## 3. Parse TEI XML: Lines, Rhyme Schemes, and Meters

Extract each line from `du_chemin_tei_texts.xml` with its piece metadata, stanza/line numbering, and per-line rhyme letter.

In [3]:
# load our files and extract lines with rhyme info into a DataFrame
tree = etree.parse("du_chemin_tei_texts.xml")
root = tree.getroot()
rows = []

for div in root.findall(f'.//{{{NS}}}body/{{{NS}}}div'):
    piece_id  = div.get(f'{{{XML_NS}}}id')
    rhyme     = div.get('rhyme', '')
    meter     = div.get('met', '')
    irregular = div.get('real') == 'irregular'

    lg_elements = div.findall(f'{{{NS}}}lg')

    if lg_elements:
        for lg_num, lg in enumerate(lg_elements, start=1):
            for l_num, l in enumerate(lg.findall(f'{{{NS}}}l'), start=1):
                rows.append({
                    'piece_id' : piece_id,
                    'rhyme'    : rhyme,
                    'meter'    : meter,
                    'irregular': irregular,
                    'lg'       : lg_num,
                    'line'     : l_num,
                    'text'     : get_line_text(l),
                })
    else:
        # Lines sit directly inside <div> with no <lg> wrapper
        for l_num, l in enumerate(div.findall(f'{{{NS}}}l'), start=1):
            rows.append({
                'piece_id' : piece_id,
                'rhyme'    : rhyme,
                'meter'    : meter,
                'irregular': irregular,
                'lg'       : 1,
                'line'     : l_num,
                'text'     : get_line_text(l),
            })

tei_lines = pd.DataFrame(rows)

# Assign line_rhyme: sequential position within each piece indexes into the rhyme string
tei_lines['line_seq']   = tei_lines.groupby('piece_id').cumcount()
tei_lines['line_rhyme'] = tei_lines.apply(
    lambda r: r['rhyme'][r['line_seq']] if r['line_seq'] < len(r['rhyme']) else '', axis=1
)
tei_lines = tei_lines.drop(columns='line_seq')
tei_lines['book'] = tei_lines['piece_id'].str[2:4].astype(int)

tei_lines

,piece_id,rhyme,meter,irregular,lg,line,text,line_rhyme,book
0,DC0101,ababbcbc,10,False,1,1,Qui souhaitez avoir tout le plaisir,a,1
1,DC0101,ababbcbc,10,False,1,2,Qu'un amy peult vouloir honnestement,b,1
2,DC0101,ababbcbc,10,False,1,3,Prenez exemple à mon chaste desir,a,1
3,DC0101,ababbcbc,10,False,1,4,Et vous mirez en mon contentement.,b,1
4,DC0101,ababbcbc,10,False,2,1,Mais qui vouldroit audacieusement,b,1
...,...,...,...,...,...,...,...,...,...
3180,DC1620,ababbb,10,False,1,2,"Si vostrɇ amour ne me donnɇ allegeance,",b,16
3181,DC1620,ababbb,10,False,1,3,"Mais de douleur vostre face jolye,",a,16
3182,DC1620,ababbb,10,False,1,4,Par quelquɇ espoir me promet delivrance.,b,16
3183,DC1620,ababbb,10,False,1,5,"Si j’ay refus j’en requerray vengeancɇ,",b,16


## 4.  Clean up of the rhyme words

- remove non-alphabetic characters from the end of the line text
- get the last word of each line, cleaned of non-alphabetic characters
- add a column for the rhyme word (last word of the line, cleaned of non-alphabetic characters) 


In [4]:
# remove non-alphabetic characters from the end of the line text
def remove_non_alpha_chars(text):
    return re.sub("\W*$", "", text)

tei_lines.text.apply(remove_non_alpha_chars).str.split()

0           [Qui, souhaitez, avoir, tout, le, plaisir]
1           [Qu'un, amy, peult, vouloir, honnestement]
2             [Prenez, exemple, à, mon, chaste, desir]
3             [Et, vous, mirez, en, mon, contentement]
4               [Mais, qui, vouldroit, audacieusement]
                             ...                      
3180    [Si, vostrɇ, amour, ne, me, donnɇ, allegeance]
3181          [Mais, de, douleur, vostre, face, jolye]
3182    [Par, quelquɇ, espoir, me, promet, delivrance]
3183     [Si, j’ay, refus, j’en, requerray, vengeancɇ]
3184    [Au, dieu, d’amour,, qui, a, toute, puissance]
Name: text, Length: 3185, dtype: object

In [5]:
# Get the last word of each line, cleaned of non-alphabetic characters

tei_lines.text.apply(remove_non_alpha_chars).str.split().str.get(-1).head()

0           plaisir
1      honnestement
2             desir
3      contentement
4    audacieusement
Name: text, dtype: object

In [6]:
# Add a column for the rhyme word (last word of the line, cleaned of non-alphabetic characters)
tei_lines["rhyme_word"] = tei_lines.text.apply(remove_non_alpha_chars).str.split().str.get(-1)
tei_lines

,piece_id,rhyme,meter,irregular,lg,line,text,line_rhyme,book,rhyme_word
0,DC0101,ababbcbc,10,False,1,1,Qui souhaitez avoir tout le plaisir,a,1,plaisir
1,DC0101,ababbcbc,10,False,1,2,Qu'un amy peult vouloir honnestement,b,1,honnestement
2,DC0101,ababbcbc,10,False,1,3,Prenez exemple à mon chaste desir,a,1,desir
3,DC0101,ababbcbc,10,False,1,4,Et vous mirez en mon contentement.,b,1,contentement
4,DC0101,ababbcbc,10,False,2,1,Mais qui vouldroit audacieusement,b,1,audacieusement
...,...,...,...,...,...,...,...,...,...,...
3180,DC1620,ababbb,10,False,1,2,"Si vostrɇ amour ne me donnɇ allegeance,",b,16,allegeance
3181,DC1620,ababbb,10,False,1,3,"Mais de douleur vostre face jolye,",a,16,jolye
3182,DC1620,ababbb,10,False,1,4,Par quelquɇ espoir me promet delivrance.,b,16,delivrance
3183,DC1620,ababbb,10,False,1,5,"Si j’ay refus j’en requerray vengeancɇ,",b,16,vengeancɇ


In [7]:

# Group by piece and rhyme letter, collecting rhyme words into lists
rhymes = tei_lines.groupby(["book", "piece_id", "line_rhyme"]).rhyme_word.apply(list).reset_index()
rhymes

,book,piece_id,line_rhyme,rhyme_word
0,1,DC0101,a,"[plaisir, desir]"
1,1,DC0101,b,"[honnestement, contentement, audacieusement, h..."
2,1,DC0101,c,"[tient, appartient]"
3,1,DC0102,a,"[plaist, desplait]"
4,1,DC0102,b,"[marchander, demander]"
...,...,...,...,...
1270,16,DC1619,a,"[dire, martyre]"
1271,16,DC1619,b,"[desplaire, satisfaire]"
1272,16,DC1619,c,[aymer]
1273,16,DC1620,a,"[vie, jolye]"


## 5.  Get the rhyme pairs
- Group by piece and rhyme letter, collecting rhyme words into lists
- get all unique pairs of rhyme words within each rhyme group
- explode the list of pairs into rows, drop any NaN values (from rhyme groups with only one word), and get unique pairs



In [8]:

# get all unique pairs of rhyme words within each rhyme group
rhyme_pairs = rhymes.rhyme_word.apply(lambda x: list(combinations(x, 2)))
rhyme_pairs.iloc[1]

[('honnestement', 'contentement'),
 ('honnestement', 'audacieusement'),
 ('honnestement', 'humainement'),
 ('contentement', 'audacieusement'),
 ('contentement', 'humainement'),
 ('audacieusement', 'humainement')]

In [9]:
# Count how many pieces each rhyme-word pair co-occurs in
pair_counts = (
    rhyme_pairs.explode()
    .dropna()
    .apply(lambda p: tuple(sorted(p)))   # normalize so (a,b) == (b,a)
    .value_counts()
)
weighted_rhyme_pairs = [(a, b, w) for (a, b), w in pair_counts.items()]
weighted_rhyme_pairs[:5]

[('noir', 'noir', 15),
 ('dire', 'martyre', 13),
 ('desire', 'dire', 8),
 ('desire', 'martyre', 7),
 ('avoir', 'voir', 7)]

## 6.  Build the rhyme network
- create a graph from the unique rhyme pairs
- visualize the graph with pyvis (you will need to load the html file in your browser to see the graph)

In [10]:
def add_communities(G):
    G = deepcopy(G)
    partition = community_louvain.best_partition(G)
    nx.set_node_attributes(G, partition, "group")
    return G

G = nx.Graph()
G.add_weighted_edges_from(weighted_rhyme_pairs)   # weight = co-occurrence count
G = add_communities(G)

# Edge width proportional to co-occurrence count (pyvis reads 'width' from nx edge data)
for u, v, data in G.edges(data=True):
    data['width'] = data['weight']

# Hover title: list every piece_id in which each rhyme word appears
word_pieces = (
    tei_lines.groupby('rhyme_word')['piece_id']
    .apply(lambda x: sorted(x.unique()))
    .to_dict()
)
titles = {node: '<br>'.join(word_pieces.get(node, [])) for node in G.nodes()}
nx.set_node_attributes(G, titles, 'title')

# Node size: proportional to how many times the word appears across all lines
word_counts = tei_lines['rhyme_word'].value_counts().to_dict()
sizes = {node: max(10, word_counts.get(node, 1) * 3) for node in G.nodes()}
nx.set_node_attributes(G, sizes, 'size')

In [11]:
# you will need to load the html file in your browser to see the graph

pyvis_graph = Network(notebook=True, width="1600px", height="900px", bgcolor="black", font_color="white")
pyvis_graph.from_nx(G)
pyvis_graph.show("rhymes.html")